# Módulo 5 · Model Context Protocol (MCP)

**Duración sugerida:** 4 horas · **Entrega:** servidor MCP funcional, configuración,
ejemplos de llamadas y evaluación de riesgos/permisos.

## Objetivos del laboratorio

- Entender el **problema que resuelve MCP** y su **arquitectura** (host, cliente, servidor).
- Crear un **servidor MCP** que exponga **herramientas** para consultar documentos/datos.
- Conectar un **cliente** y comprobar **descubrimiento e invocación**.
- Discutir **seguridad**: mínimo privilegio, control de acceso y revisión de operaciones sensibles.

## Conexión con el proyecto ARAS

En ARAS, MCP da **acceso controlado** a documentos y fuentes de datos autorizadas.
En lugar de dar al modelo "todo el sistema de archivos", exponemos solo las
herramientas necesarias.

## 1. Arquitectura

- **Host:** la aplicación que orquesta (tu notebook / app).
- **Cliente:** se conecta a uno o más servidores.
- **Servidor:** expone **tools** (funciones), **resources** (datos) y **prompts** (plantillas).

MCP estandariza el **descubrimiento** y la **invocación**, evitando integraciones
a medida por cada servicio.

## 2. Dependencias

In [ ]:
%pip install -q mcp

## 3. Crear un servidor MCP

Escribimos el servidor en un archivo con `%%writefile`. Expone dos herramientas
sobre una colección local de documentos.

In [ ]:
%%writefile mcp_doc_server.py
import json
from mcp.server.mcpserver import MCPServer

# El nombre identifica al servidor durante el descubrimiento.
server = MCPServer(name="servidor-documentos-aras")

DOCUMENTOS = {
    "mcp": "Model Context Protocol (MCP) estandariza cómo los agentes acceden a herramientas y recursos externos.",
    "rag": "Retrieval-Augmented Generation combina búsqueda en documentos con generación para reducir alucinaciones.",
    "langgraph": "LangGraph permite construir grafos de estado para pipelines, agentes y sistemas multiagente.",
    "evaluacion": "La evaluación de agentes mide relevancia, uso correcto de herramientas y trazabilidad.",
}

@server.tool()
def buscar_documentos(consulta: str) -> str:
    """Busca fragmentos relevantes en la colección local de documentos."""
    resultados = [texto for texto in DOCUMENTOS.values() if consulta.lower() in texto.lower()]
    if not resultados:
        return "No se encontraron documentos relevantes."
    return "\n---\n".join(resultados)

@server.tool()
def listar_documentos() -> str:
    """Devuelve la lista de identificadores de documentos disponibles."""
    return json.dumps(list(DOCUMENTOS.keys()), ensure_ascii=False)

if __name__ == "__main__":
    # Transporte stdio: el cliente lanza el proceso y se comunica por entrada/salida estándar.
    server.run()

## 4. Cliente: descubrir herramientas

El cliente lanza el servidor como subproceso (`stdio`) y lista las herramientas disponibles.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def demo_descubrimiento():
    parametros = StdioServerParameters(command="python", args=["mcp_doc_server.py"])
    async with stdio_client(parametros) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            herramientas = await session.list_tools()
            print("Herramientas descubiertas:")
            for t in herramientas.tools:
                print(f"  - {t.name}: {t.description}")

await demo_descubrimiento()

## 5. Cliente: invocar herramientas

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def demo_invocacion():
    parametros = StdioServerParameters(command="python", args=["mcp_doc_server.py"])
    async with stdio_client(parametros) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            print("=== listar_documentos ===")
            r1 = await session.call_tool("listar_documentos", {})
            print(r1.content[0].text)

            print("\n=== buscar_documentos('MCP') ===")
            r2 = await session.call_tool("buscar_documentos", {"consulta": "MCP"})
            print(r2.content[0].text)

            print("\n=== buscar_documentos('inexistente') ===")
            r3 = await session.call_tool("buscar_documentos", {"consulta": "blockchain"})
            print(r3.content[0].text)

await demo_invocacion()

## 6. Recursos y prompts (breve)

Además de herramientas, MCP define:

- **Resources:** datos expuestos como URI legibles por el cliente.
- **Prompts:** plantillas reutilizables definidas por el servidor.

En este laboratorio nos centramos en herramientas; puedes añadir un recurso con
`@mcp.resource("documentos://lista")` que devuelva la lista de documentos.

## 7. Seguridad y permisos

Principios que aplica ARAS:

1. **Mínimo privilegio:** exponer solo las herramientas necesarias (no "todo el sistema").
2. **Control de acceso:** validar quién puede invocar operaciones sensibles.
3. **Validación de entradas:** rechazar argumentos malformados o peligrosos.
4. **Revisión de operaciones sensibles:** exigir confirmación humana para acciones destructivas.
5. **Registro (auditoría):** trazar qué herramienta se llamó y con qué argumentos.

In [ ]:
# Ejemplo de validación de entradas en el servidor.
def consulta_valida(consulta: str) -> bool:
    if len(consulta) > 200:
        return False
    if any(c in consulta for c in "<>{}"):
        return False
    return True

for c in ["MCP", "x" * 300, "DROP <table>"]:
    print(f"'{c[:20]}...' -> válida: {consulta_valida(c)}")

## Resumen del laboratorio

- Creamos un **servidor MCP** (`MCPServer`, paquete `mcp` v2) con dos herramientas sobre documentos.
- Conectamos un **cliente** por `stdio`, **descubrimos** e **invocamos** las herramientas.
- Discutimos **seguridad**: mínimo privilegio, validación de entradas y revisión.

**Entregable del Módulo 5:** servidor MCP funcional, instrucciones de configuración,
ejemplos de llamadas y breve evaluación de riesgos y permisos.